# ViralCutter
Uma alternativa gratuita ao `opus.pro` e ao `vidyo.ai`

# Suporte em:
[![](https://dcbadge.limes.pink/api/server/tAdPHFAbud)](https://discord.gg/tAdPHFAbud)

# TODO📝
- [x] Release code
- [ ] Huggingface SpaceDemo
- [x] Two face in the cut
- [x] Custom caption and burn
- [x] Make the code faster
- [ ] More types of framing beyond 9:16

In [ ]:
#@title Installation
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kodokbakar/ViralCutter.git"
BRANCH = "dev"

ROOT = Path("/content/ViralCutter")
VENV = Path("/content/viralcutter-venv")
PYTHON = VENV / "bin/python"
INSTALL_LOG = Path("/content/viralcutter-install.log")

BASE_ENV = os.environ.copy()
BASE_ENV.update(
    {
        "MPLBACKEND": "Agg",
        "UV_LINK_MODE": "copy",
        "PYTHONSAFEPATH": "1",
    }
)


def run(*command, cwd=None):
    command = [str(part) for part in command]

    result = subprocess.run(
        command,
        cwd=cwd,
        env=BASE_ENV.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    print(result.stdout)

    if result.returncode != 0:
        INSTALL_LOG.write_text(result.stdout, encoding="utf-8")
        raise RuntimeError(
            f"Command failed: {' '.join(command)}\n"
            f"Full log: {INSTALL_LOG}"
        )


print("Cleaning previous installation...")
shutil.rmtree(ROOT, ignore_errors=True)
shutil.rmtree(VENV, ignore_errors=True)
INSTALL_LOG.unlink(missing_ok=True)

print("Installing system packages...")
run("apt-get", "update", "-qq")
run(
    "apt-get",
    "install",
    "-y",
    "-qq",
    "ffmpeg",
    "xvfb",
    "pkg-config",
    "build-essential",
    "python3-dev",
    "libavformat-dev",
    "libavcodec-dev",
    "libavdevice-dev",
    "libavutil-dev",
    "libavfilter-dev",
    "libswscale-dev",
    "libswresample-dev",
)

print("Installing uv...")
run(
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "uv",
)

print(f"Cloning branch: {BRANCH}")
run(
    "git",
    "clone",
    "--depth",
    "1",
    "--single-branch",
    "--branch",
    BRANCH,
    REPO_URL,
    ROOT,
)

print("Creating Python 3.11 environment...")
run(
    "uv",
    "venv",
    "--python",
    "3.11",
    VENV,
)

print("Installing CUDA PyTorch...")
run(
    "uv",
    "pip",
    "install",
    "--python",
    PYTHON,
    "--index-url",
    "https://download.pytorch.org/whl/cu121",
    "torch==2.3.1+cu121",
    "torchaudio==2.3.1+cu121",
)

print("Installing application requirements...")
run(
    "uv",
    "pip",
    "install",
    "--python",
    PYTHON,
    "--index-url",
    "https://pypi.org/simple",
    "-r",
    ROOT / "requirements.txt",
)

print("Checking dependency consistency...")
run(
    "uv",
    "pip",
    "check",
    "--python",
    PYTHON,
)

print("Verifying critical imports...")
run(
    PYTHON,
    "-P",
    "-c",
    (
        "import os, sys; "
        "print('Python:', sys.executable); "
        "print('Working directory:', os.getcwd()); "
        "print('MPLBACKEND:', os.environ.get('MPLBACKEND')); "
        "import regex, defusedxml, nltk; "
        "import numpy, torch, av, ctranslate2, faster_whisper, whisperx; "
        "import transformers, huggingface_hub, gradio, gradio_client; "
        "from google import genai; "
        "print('regex:', regex.__file__); "
        "print('defusedxml:', defusedxml.__file__); "
        "print('NLTK:', nltk.__version__); "
        "print('Torch:', torch.__version__); "
        "print('CUDA available:', torch.cuda.is_available()); "
        "print('WhisperX import: OK'); "
        "print('Gradio:', gradio.__version__); "
        "print('Gradio Client:', gradio_client.__version__); "
        "print('Critical imports: OK')"
    ),
    cwd=ROOT,
)

print("Installation completed.")
print("Branch:", BRANCH)
print("Project:", ROOT)
print("Python:", PYTHON)

In [ ]:
#@title Runtime Doctor Preflight
import os
import subprocess
from pathlib import Path

ROOT = Path("/content/ViralCutter")
PYTHON = Path("/content/viralcutter-venv/bin/python")
DOCTOR = ROOT / "webui/runtime_doctor.py"

env = os.environ.copy()
env.update(
    {
        "MPLBACKEND": "Agg",
        "PYTHONSAFEPATH": "1",
        "PYTHONUNBUFFERED": "1",
    }
)

result = subprocess.run(
    [
        str(PYTHON),
        "-P",
        "-u",
        str(DOCTOR),
    ],
    cwd=ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(
        "Runtime Doctor found blocking issues. "
        "Review the failed check printed above."
    )

print("Runtime Doctor passed.")

In [ ]:
#@title Configuration and Run
import os
import subprocess
from pathlib import Path
import time
os.environ["MPLBACKEND"] = "Agg"

from google.colab import drive

ROOT = Path("/content/ViralCutter")
PYTHON = Path("/content/viralcutter-venv/bin/python")
OUTPUT_DIR = Path("/content/drive/MyDrive/ViralCutter/VIRALS")

print("Mounting Google Drive...")
drive.mount("/content/drive", force_remount=False)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["pkill", "-f", "Xvfb :1"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

xvfb = subprocess.Popen(
    [
        "Xvfb",
        ":1",
        "-screen",
        "0",
        "2560x1440x24",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

time.sleep(0.5)

if xvfb.poll() is not None:
    raise RuntimeError("Xvfb failed to start on display :1.")

env = os.environ.copy()
env.update(
    {
        "DISPLAY": ":1",
        "MPLBACKEND": "Agg",
        "VIRALCUTTER_OUTPUT_DIR": str(OUTPUT_DIR),
        "PYTHONUNBUFFERED": "1",
    }
)

print("Output directory:", OUTPUT_DIR)
subprocess.run(
    [
        str(PYTHON),
        "-c",
        (
            "import gradio, gradio_client; "
            "print('Gradio:', gradio.__version__); "
            "print('Gradio Client:', gradio_client.__version__)"
        ),
    ],
    cwd=ROOT,
    env=env,
    check=True,
)
print("Starting ViralCutter...")

process = subprocess.Popen(
    [
        str(PYTHON),
        "-u",
        "webui/app.py",
        "--colab",
    ],
    cwd=ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

exit_code = process.wait()

if exit_code:
    raise RuntimeError(
        f"ViralCutter exited with status {exit_code}. "
        "The actual traceback is printed above."
    )

#Créditos

Inspirado no [reels clips automator](https://github.com/eddieoz/reels-clips-automator) e no [YoutubeVideoToAIPoweredShorts](https://github.com/Fitsbit/YoutubeVideoToAIPoweredShorts)<br>

---
![Rafa.png](https://i.imgur.com/cGknQpU.png;base64)

Desenvolvido por **Rafa.Godoy**<br>
[ ![GitHub](https://img.shields.io/badge/github-%23121011.svg?style=for-the-badge&logo=github&logoColor=white) ](https://github.com/rafaelGodoyEbert)<br>
[ ![X](https://img.shields.io/twitter/url?url=https%3A%2F%2Ftwitter.com%2FGodoyEbert) ](https://twitter.com/GodoyEbert)<br>
[Instagram](https://www.instagram.com/rafael.godoy.ebert/)<br>
[ ![](https://dcbadge.vercel.app/api/server/aihubbrasil) ](https://discord.gg/aihubbrasil)

`0.1v Alpha`<br>

Apenas uma alternativa gratuita ao `opus.pro` e ao `vidyo.ai`<br>
